# Course 1: Geometric Deep Learning and Graphs
## Session 7: E(3)-equivariant models for atoms

**Goal:** write an EGNN layer, check its symmetries with numbers, and train it on a small invariant target.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) on small built-in or synthetic data. Cells labelled NOT RUN need a library, a dataset download or a GPU; run them on Kaggle or Colab.

### 1. The EGNN layer
Messages use only distances (invariant). Positions are updated along the difference vectors (equivariant).

- message: `m_ij = phi_e(h_i, h_j, |x_i - x_j|²)`
- positions: `x_i <- x_i + mean_j (x_i - x_j) * phi_x(m_ij)`
- features: `h_i <- h_i + phi_h(h_i, sum_j m_ij)`

In [ ]:
import numpy as np, torch, torch.nn as nn

class EGNNLayer(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.phi_e = nn.Sequential(nn.Linear(2 * d + 1, d), nn.SiLU(), nn.Linear(d, d), nn.SiLU())
        self.phi_x = nn.Sequential(nn.Linear(d, d), nn.SiLU(), nn.Linear(d, 1))
        self.phi_h = nn.Sequential(nn.Linear(2 * d, d), nn.SiLU(), nn.Linear(d, d))
    def forward(self, h, x):                              # h: (B, N, d)   x: (B, N, 3)
        B, N, _ = x.shape
        diff = x[:, :, None] - x[:, None]                 # (B, N, N, 3)
        dist2 = (diff ** 2).sum(-1, keepdim=True)
        m = self.phi_e(torch.cat([h[:, :, None].expand(B, N, N, -1), h[:, None].expand(B, N, N, -1), dist2], -1))
        mask = 1 - torch.eye(N)[None, :, :, None]         # no message to yourself
        x = x + (diff * self.phi_x(m) * mask).sum(2) / (N - 1)
        h = h + self.phi_h(torch.cat([h, (m * mask).sum(2)], -1))
        return h, x

class EGNN(nn.Module):
    def __init__(self, d=32, layers=3):
        super().__init__()
        self.embed = nn.Parameter(torch.randn(d) * 0.1)
        self.layers = nn.ModuleList([EGNNLayer(d) for _ in range(layers)])
        self.out = nn.Sequential(nn.Linear(d, d), nn.SiLU(), nn.Linear(d, 1))
    def forward(self, x):
        h = self.embed.expand(x.shape[0], x.shape[1], -1)
        for layer in self.layers: h, x = layer(h, x)
        return self.out(h).sum(1).squeeze(-1), x          # an invariant scalar, and equivariant positions

### 2. Check the symmetries
Rotate and shift the input. The scalar must not change, and the output positions must rotate and shift the same way.

In [ ]:
torch.manual_seed(0)
model = EGNN().double()
x = torch.randn(2, 5, 3, dtype=torch.double)
Q, _ = torch.linalg.qr(torch.randn(3, 3, dtype=torch.double))        # a random orthogonal matrix
t = torch.randn(1, 1, 3, dtype=torch.double)
e1, x1 = model(x)
e2, x2 = model(x @ Q.T + t)
print("invariance error   :", (e1 - e2).abs().max().item())
print("equivariance error :", (x1 @ Q.T + t - x2).abs().max().item())
assert torch.allclose(e1, e2, atol=1e-9) and torch.allclose(x1 @ Q.T + t, x2, atol=1e-9)

### 3. Train on a toy 'energy'
Target: the sum over atom pairs of `exp(-distance)`. It depends on shape only, like a real molecular energy. A plain MLP on raw coordinates is the baseline.

In [ ]:
def energy(x):
    d = torch.cdist(x, x); iu = torch.triu_indices(x.shape[1], x.shape[1], 1)
    return torch.exp(-d[:, iu[0], iu[1]]).sum(1)

torch.manual_seed(0)
Xtr, Xte = torch.randn(512, 5, 3), torch.randn(256, 5, 3)
ytr, yte = energy(Xtr), energy(Xte)
mu, sd = ytr.mean(), ytr.std()
Q, _ = torch.linalg.qr(torch.randn(3, 3)); Xte_rot = Xte @ Q.T      # the same test molecules, rotated

def fit(forward, params, epochs=60):
    opt = torch.optim.Adam(params, lr=2e-3)
    for _ in range(epochs):
        for idx in torch.randperm(512).split(64):
            loss = ((forward(Xtr[idx]) - (ytr[idx] - mu) / sd) ** 2).mean(); opt.zero_grad(); loss.backward(); opt.step()
    mae = lambda X: ((forward(X) * sd + mu) - yte).abs().mean().item()
    with torch.no_grad(): return mae(Xte), mae(Xte_rot)

egnn = EGNN(); res_egnn = fit(lambda X: egnn(X)[0], egnn.parameters())
mlp = nn.Sequential(nn.Flatten(), nn.Linear(15, 128), nn.SiLU(), nn.Linear(128, 128), nn.SiLU(), nn.Linear(128, 1))
res_mlp = fit(lambda X: mlp(X).squeeze(-1), mlp.parameters())
print("baseline (predict the mean): MAE %.3f" % (yte - ytr.mean()).abs().mean().item())
print("MLP on coordinates : test MAE %.3f   rotated test MAE %.3f" % res_mlp)
print("EGNN               : test MAE %.3f   rotated test MAE %.3f" % res_egnn)
assert res_egnn[0] < res_mlp[0] and abs(res_egnn[0] - res_egnn[1]) < 1e-3

### 4. QM9 with an EGNN (NOT RUN here)

In [ ]:
# NOT RUN: needs torch_geometric, the QM9 download (about 130k molecules) and a GPU.
from torch_geometric.datasets import QM9
ds = QM9("data/qm9")
target = 7                                    # index 7 is U0, the internal energy at 0 K; check the dataset card
# Each item has ds[i].z (atomic numbers), ds[i].pos (3-D positions) and ds[i].y (19 targets).
# Steps: (1) replace `self.embed` with nn.Embedding(10, d) indexed by z, (2) batch molecules with padding and a mask,
# (3) standardize the target with training-set statistics, (4) train with L1 loss, (5) report MAE in the paper's units.

### TODO
1. Break the model on purpose: add `x.sum(-1)` to the message input. Which check in section 2 fails, and why?
2. Predict forces as the negative gradient of the predicted energy with respect to `x` using `torch.autograd.grad`. Check they rotate with the molecule.
3. Run QM9 for one target and compare your MAE with the EGNN paper's table.